# 병합 충돌 해결

모든 저장소는 임시 폴더에 만들며 실제 GitHub 계정이나 원격 저장소를 변경하지 않습니다.

## 명령을 실행하기 전에 알아둘 Git 상태

Git은 파일을 세 자리에서 봅니다. **작업 폴더(working tree)** 는 지금 편집하는 파일이 있는 자리입니다. **색인(index)** 은 다음 커밋에 넣을 파일 상태를 골라 두는 자리이며, 스테이징 영역(staging area)이라고도 합니다. **커밋(commit)** 은 색인에 고른 상태를 저장소 이력에 기록한 결과입니다. 따라서 `git add app.py`는 파일을 영구 저장하는 명령이 아니라 현재 `app.py`를 색인에 올리는 명령입니다. 이어서 `git commit -m "메시지"`를 실행해야 그 상태가 커밋으로 기록됩니다.

**HEAD** 는 현재 체크아웃한 커밋을 가리킵니다. 보통 HEAD는 `main` 같은 **브랜치(branch)** 이름을 가리키고, 브랜치 이름은 다시 최신 커밋을 가리킵니다. 새 커밋을 만들면 현재 브랜치가 새 커밋으로 이동합니다. 이 절에서는 현재 브랜치의 HEAD와 `origin/main`을 비교해 로컬과 원격의 갈라진 이력을 읽습니다.

명령은 왼쪽부터 `git` → 하위 명령 → 옵션 → 대상으로 읽습니다. 예를 들어 `git status --short`에서 `status`는 상태를 보고, `--short`는 짧은 형식으로 표시하라는 옵션입니다. 코드의 `git("status", "--short", cwd=repo)`는 터미널에서 같은 명령을 실행하도록 인자를 나눈 Python 표현입니다.


## 준비

공용 원격과 두 작업 폴더를 만들고 출고 기준을 기록합니다. **병합(merge)** 은 갈라진 두 커밋 흐름의 변경을 하나의 결과로 합치는 작업입니다. 서로 다른 파일이나 서로 다른 줄을 바꿨다면 Git이 자동으로 합칠 수 있습니다. 같은 파일의 같은 줄을 서로 다르게 바꾸면 어느 내용을 선택할지 알 수 없어 **병합 충돌(merge conflict)** 로 멈춥니다.

이 실습은 `message.txt` 한 줄을 Alice와 Bob이 서로 다르게 수정합니다. `policy.txt`는 정답을 자동 생성하는 파일이 아니라 사람이 최종 문장을 판단할 근거입니다. 충돌 해결은 표시를 지우는 작업만이 아니라 두 변경의 의도를 확인해 올바른 파일 내용을 새로 결정하는 과정입니다.

In [1]:
# 셀이 만든 저장소만 사용하는 임시 작업 폴더를 준비합니다.
import subprocess
import sys
import tempfile
from pathlib import Path
workspace = tempfile.TemporaryDirectory(prefix="git-lesson-")
root = Path(workspace.name)
# cwd는 명령을 실행할 폴더이며 check=True는 명령 실패를 예외로 알립니다.
def git(*args: str, cwd: Path | None = None, check: bool = True) -> str:
    """지정한 임시 저장소에서 Git 명령을 실행합니다."""
    result = subprocess.run(["git", *args], cwd=cwd or root,
                            text=True, capture_output=True, check=check)
    return result.stdout.strip() or result.stderr.strip()
def configure(repo: Path) -> None:
    """실습 저장소에만 작성자 정보를 설정합니다."""
    git("config", "user.name", "Git Learner", cwd=repo)
    git("config", "user.email", "learner@example.invalid", cwd=repo)

In [2]:
# 두 작업자가 함께 사용할 원격을 준비합니다.
remote = root / "team.git"
git("init", "--bare", "--initial-branch=main", remote.as_posix())
alice, bob = root / "alice", root / "bob"
git("clone", remote.as_posix(), alice.as_posix())
configure(alice)
(alice / "policy.txt").write_text(
    "14시 이전 결제: 오늘 출고\n14시 이후 결제: 내일 출고\n", encoding="utf-8"
)
(alice / "message.txt").write_text("배송 안내: 준비 중\n", encoding="utf-8")
git("add", "policy.txt", "message.txt", cwd=alice)
git("commit", "-m", "docs: 출고 기준 기록", cwd=alice)
git("push", "-u", "origin", "main", cwd=alice)
# Bob은 Alice의 두 번째 변경이 게시되기 전에 공통 시작점을 복제합니다.
git("clone", remote.as_posix(), bob.as_posix())
configure(bob)
print("공용 원격과 Bob의 공통 시작점 준비 완료")

공용 원격과 Bob의 공통 시작점 준비 완료


## 기본 실행

Alice가 정책을 반영한 안내를 먼저 원격에 게시합니다.

In [3]:
# Alice가 수정한 같은 줄을 먼저 원격 main에 반영합니다.
(alice / "message.txt").write_text("배송 안내: 14시 이전은 오늘 출고\n", encoding="utf-8")
git("add", "message.txt", cwd=alice)
git("commit", "-m", "docs: 당일 출고 안내", cwd=alice)
git("push", cwd=alice)
print(git("log", "--oneline", "-2", cwd=alice))

d7e1e77 docs: 당일 출고 안내
e08f056 docs: 출고 기준 기록


## 값 하나를 바꿔 비교

Bob은 Alice가 두 번째 커밋을 push하기 전에 복제한 공통 `main`에서 같은 줄을 다른 문장으로 수정합니다. 따라서 두 사람은 같은 부모 커밋에서 자연스럽게 갈라지며 브랜치 위치를 따로 옮길 필요가 없습니다.

Bob의 push가 `non-fast-forward`로 거절되는 이유는 원격 `main`에 Bob이 갖고 있지 않은 Alice의 커밋이 있기 때문입니다. 강제 push로 덮는 대신 `git pull --no-rebase origin main`으로 원격 커밋을 가져와 병합합니다. `--no-rebase`는 가져온 커밋 위로 Bob의 커밋을 다시 만들지 않고 두 흐름을 병합하라는 뜻입니다.

In [4]:
# Bob이 공통 시작점에서 같은 줄을 다르게 수정해 별도 커밋을 만듭니다.
(bob / "message.txt").write_text("배송 안내: 내일 출고\n", encoding="utf-8")
git("add", "message.txt", cwd=bob)
git("commit", "-m", "docs: 익일 출고 안내", cwd=bob)
push = subprocess.run(["git", "push", "origin", "main"], cwd=bob,
                      text=True, capture_output=True)
print("push 종료 코드:", push.returncode)
print("push 거절:", push.returncode != 0)
print("거절 이유 끝줄:", push.stderr.strip().splitlines()[-1])
assert push.returncode != 0

push 종료 코드: 1
push 거절: True
거절 이유 끝줄: hint: See the 'Note about fast-forwards' in 'git push --help' for details.


## 실패 재현

Bob이 원격 변경을 가져와 병합하면 같은 줄의 차이에서 충돌합니다.

In [5]:
# --no-rebase로 원격 main을 병합하면 같은 줄의 충돌 표시가 남습니다.
merge = subprocess.run(
    ["git", "pull", "--no-rebase", "origin", "main"], cwd=bob,
    text=True, capture_output=True)
print("종료 코드:", merge.returncode)
print(git("status", "--short", cwd=bob))
print((bob / "message.txt").read_text(encoding="utf-8"))
assert merge.returncode != 0
assert "<<<<<<<" in (bob / "message.txt").read_text(encoding="utf-8")

종료 코드: 1
UU message.txt
<<<<<<< HEAD
배송 안내: 내일 출고
배송 안내: 14시 이전은 오늘 출고
>>>>>>> d7e1e77c2eb32dc159f0af8c67a396477e7ce7fe



## 수정

저장소의 정책 문서를 근거로 안내를 확정하고 원격에 게시합니다.

In [6]:
# 정책에 맞춘 실제 파일에서 충돌 표시 제거와 병합 완료를 확인합니다.
resolved = "배송 안내: 14시 이전 결제는 오늘 출고, 14시 이후 결제는 내일 출고\n"
(bob / "message.txt").write_text(resolved, encoding="utf-8")
git("add", "message.txt", cwd=bob)
git("commit", "-m", "merge: 출고 정책에 맞춰 충돌 해결", cwd=bob)
git("push", "origin", "main", cwd=bob)
print((bob / "policy.txt").read_text(encoding="utf-8").strip())
print((bob / "message.txt").read_text(encoding="utf-8").strip())
parents = git("show", "--pretty=%P", "--no-patch", "HEAD", cwd=bob)
print("병합 부모:", parents)
saved = (bob / "message.txt").read_text(encoding="utf-8")
assert all(marker not in saved for marker in ("<<<<<<<", "=======", ">>>>>>>"))
assert not git("status", "--porcelain", cwd=bob)
assert len(parents.split()) == 2
assert git("rev-parse", "HEAD", cwd=bob) == git("rev-parse", "origin/main", cwd=bob)

14시 이전 결제: 오늘 출고
14시 이후 결제: 내일 출고
배송 안내: 14시 이전 결제는 오늘 출고, 14시 이후 결제는 내일 출고
병합 부모: 589342bfdcd51bd054c2ea92c3c43359b0a29b43 d7e1e77c2eb32dc159f0af8c67a396477e7ce7fe
